In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Using device:", device)

PyTorch version: 2.12.0
MPS available: True
Using device: mps


In [2]:
from datasets import load_dataset

# pulls directly from huggingface.co/datasets/imdb
dataset = load_dataset("imdb")

print(dataset)
print("\nOne example:")
print(dataset["train"][0])

/Users/victorhugo/Documents/Work/Skills/AI Engineering/Fine Tuning/sentiment-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

One example:
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking p

# Tokenization

In [3]:
from transformers import AutoTokenizer

# load DistilBERT's tokenizer from Hugging Face
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# tokenize one sentence
text = "This movie was absolutely brilliant"
tokens = tokenizer(text)

print("Token IDs:", tokens["input_ids"])
print("Attention mask:", tokens["attention_mask"])

# see the actual tokens (words/subwords)
print("Tokens:", tokenizer.convert_ids_to_tokens(tokens["input_ids"]))

Token IDs: [101, 2023, 3185, 2001, 7078, 8235, 102]
Attention mask: [1, 1, 1, 1, 1, 1, 1]
Tokens: ['[CLS]', 'this', 'movie', 'was', 'absolutely', 'brilliant', '[SEP]']


The token id are fixed , always the same (frozen dictionary)


Tokenization is converting raw text into a sequence of IDs from the frozen dictionary -> Lookup step

- CLS -> Added to the start , summary token
- SEP -> Added to the end of the sentence
- Attention mask -> Pay attention to all the words of the sentence. could pad small sentences with 0 

In [4]:
weird_text = "This movie was unbelievably magnificent"
tokens = tokenizer(weird_text)
print(tokenizer.convert_ids_to_tokens(tokens["input_ids"]))

['[CLS]', 'this', 'movie', 'was', 'un', '##bel', '##ie', '##va', '##bly', 'magnificent', '[SEP]']


GPU need same length rows 

GPU processes rows in parallel, in a grid, it needs a clean matrix to do parallel math. so the 0s padding and truncate help with that

Padding fill up tokens that didn't achieve 512 tokens 

Truncate cut tokens too long to fit into 512 tokens

In [5]:
# Now tokenizer all dataset (training reviews)

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',  # pad short sentences to same length
        truncation= True,      # cut sentences longer than 512 tokens
        max_length= 512
    )
    

# MAP -> Apply to EVERY ROW
tokenized_dataset = dataset.map(tokenize_function, batched=True)  # Batched to process many reviews at once 

print(tokenized_dataset)
print("\nOne tokenized example keys:", tokenized_dataset["train"][0].keys())

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 50000
    })
})

One tokenized example keys: dict_keys(['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'])


each row = 512 token IDs (just numbers, like addresses)

review 1 → [101, 2023, 3185, 0, 0, ...]  ← 512 IDs
review 2 → [101, 1996, 8235, 0, 0, ...]  ← 512 IDs

In [6]:
# remove columns the model doesn't need
tokenized_dataset = tokenized_dataset.remove_columns(["text", "token_type_ids"])

# rename label to labels (what PyTorch expects)
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")

# tell it to return PyTorch tensors
tokenized_dataset.set_format("torch")

print(tokenized_dataset)
print("\nOne example:")
print(tokenized_dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 50000
    })
})

One example:
{'labels': tensor(0), 'input_ids': tensor([  101,  1045, 12524,  1045,  2572,  8025,  1011,  3756,  2013,  2026,
         2678,  3573,  2138,  1997,  2035,  1996,  6704,  2008,  5129,  2009,
         2043,  2009,  2001,  2034,  2207,  1999,  3476,  1012,  1045,  2036,
         2657,  2008,  2012,  2034,  2009,  2001,  8243,  2011,  1057,  1012,
         1055,  1012,  8205,  2065,  2009,  2412,  2699,  2000,  4607,  2023,
         2406,  1010,  3568,  2108,  1037,  5470,  1997,  3152,  2641,  1000,
         6801,  1000,  1045,  2428,  2018,  2000,  2156,  2023,  2005,  2870,
         1012,  1026,  7987,  1013,  1028, 

- Labels -> The target for each sentence
- input_id -> Identification for each token
- attention -> Where bert will use self attention ( which tokens influence each other )

# Load DistilBERT

In [7]:
from transformers import AutoModelForSequenceClassification
import torch

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2        # positive or negative
)

# send model to Apple GPU
device = "mps" if torch.backends.mps.is_available() else "cpu"
model = model.to(device)

print(model.config.id2label)
print(f"\nModel loaded on: {device}")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9585.89it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{0: 'LABEL_0', 1: 'LABEL_1'}

Model loaded on: mps


You loaded a pretrained DistilBERT that was trained for a different task (masked language modeling — predicting missing words). You're now repurposing it for classification.

- Original DistilBERT task:
- "The movie was [MASK]"  →  predicts "great", "terrible", etc.
-                ↑
-          needs vocab_transform, vocab_projector to output words
-         we don't need this → thrown away


The missing weights are the new layers WE added to the task , ready to learn

pre classifier layer ( new randomly initiated)
classifier layer (output 0 or 1), new randomly initiated

1. KEPT   → all 66M parameters that understand language ✅
2. THROWN → vocab layers for word prediction ❌ (wrong task)
3. ADDED  → 2 new classification layers, randomly initialized 🆕

BEFORE pre_classifier:
768 numbers = [grammar info, topic info, sentiment info, noise, noise...]

AFTER pre_classifier:
768 numbers = [sentiment info, more sentiment info, relevant patterns...]

1. FORWARD PASS  → feed data through model → get prediction
2. LOSS          → how wrong was the prediction?
3. BACKWARD PASS → adjust weights to be less wrong next time

1. loss → flows back → adjusts classifier weights
2.     → flows back → adjusts pre_classifier weights
3.     → flows back → adjusts DistilBERT weights (slightly)

# Training 

first configuration of training process

than training

In [8]:
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
import time


# 1. Define metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)   # Biggest probability index of the array
    # axis = -1 to find the biggest index across the last dimension ( not all rows, just per row)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions)
    }

# 2. Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,           # how many times to go through training data
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,           # small → we don't want to destroy pretrained weights
    eval_strategy="epoch",        # evaluate after each epoch
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=100,
    report_to="tensorboard",        #  Tensorboard

)

# 3. Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
)


# record time
start_time = time.time()
trainer.train()
end_time = time.time()

total_seconds = end_time - start_time
print(f"\n⏱️ Total training time: {total_seconds/60:.2f} minutes")
print(f"⏱️ Per epoch: {total_seconds/2/60:.2f} minutes")

print("Trainer ready ✅")

/Users/victorhugo/Documents/Work/Skills/AI Engineering/Fine Tuning/sentiment-env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.216855,0.200694,0.923000,0.920772
2,0.140314,0.229808,0.932520,0.932834


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.87it/s]
/Users/victorhugo/Documents/Work/Skills/AI Engineering/Fine Tuning/sentiment-env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.08it/s]



⏱️ Total training time: 167.71 minutes
⏱️ Per epoch: 83.86 minutes
Trainer ready ✅


In [9]:
# save everything
model.save_pretrained("./my-sentiment-model")
tokenizer.save_pretrained("./my-sentiment-model")

print("Model saved ✅")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.71it/s]

Model saved ✅


The model — all the weights, every parameter, the knowledge:

The tokenizer — the frozen dictionary:


tokenizer  → NOTHING changed
            same dictionary as before
            word "great" is still ID 2307
            training never touches it

model      → weights UPDATED
            pre_classifier learned sentiment patterns
            classifier learned positive vs negative
            DistilBERT layers slightly adjusted

# Predict

In [10]:
def predict(text):
    # 1. tokenize the input text
    inputs = tokenizer(
        text,
        return_tensors="pt",        # return PyTorch tensors
        padding=True,
        truncation=True,
        max_length=512
    )

    # 2. move inputs to same device as model
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # 3. forward pass — no gradient needed for inference
    with torch.no_grad():
        outputs = model(**inputs)

    # 4. get probabilities
    probs = torch.softmax(outputs.logits, dim=-1)
    prediction = torch.argmax(probs, dim=-1).item()

    labels = {0: "NEGATIVE ❌", 1: "POSITIVE ✅"}
    confidence = probs[0][prediction].item()

    print(f"Text:       {text}")
    print(f"Prediction: {labels[prediction]}")
    print(f"Confidence: {confidence:.1%}")
    print()

In [11]:
predict("This movie was absolutely fantastic, I loved every minute")
predict("Terrible plot, bad acting, complete waste of time")
predict("It was okay I guess, nothing special")
predict("One of the best films I have ever seen in my life")
predict("So bad it was actually good, weirdly enjoyable")

Text:       This movie was absolutely fantastic, I loved every minute
Prediction: POSITIVE ✅
Confidence: 99.3%

Text:       Terrible plot, bad acting, complete waste of time
Prediction: NEGATIVE ❌
Confidence: 99.4%

Text:       It was okay I guess, nothing special
Prediction: NEGATIVE ❌
Confidence: 92.8%

Text:       One of the best films I have ever seen in my life
Prediction: POSITIVE ✅
Confidence: 99.4%

Text:       So bad it was actually good, weirdly enjoyable
Prediction: POSITIVE ✅
Confidence: 93.0%

